# Hultgren et al. (2025)

Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

The output folder and filename convention match the original workflow. Existing files are skipped (`OVERWRITE = False`). SSP245 and SSP370 and the four original reporting periods are retained. County aggregation uses cosine-latitude area correction, with no population, LandScan, or crop weights.

Growing season: March 1 through August 31.

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    annual_sum,
    bounded_degree_days,
    degree_days_above_threshold,
    select_growing_season,
)

## 2. Paper settings

In [2]:
HULTGREN_SEASON_START = '03-01'
HULTGREN_SEASON_END = '08-31'
PAPER_NAME = 'Hultgren et al. (2025)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr')
OVERWRITE = False

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [3]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Hultgren et al. (2025)
Eligible runs: 156
Existing files to skip: 0
Runs to process: 156
Counties: 102


## 4. Paper indicator calculation

In [ ]:
def calculate_hultgren(inputs):
    """
    Calculate Hultgren et al. indicators and aggregate them
    to Illinois counties.

    Output dimensions:
        year × county
    """
    required_variables = {'tasmin', 'tasmax', 'pr'}
    missing_variables = required_variables - set(inputs)
    if missing_variables:
        raise KeyError(f'Missing required Hultgren input variables: {sorted(missing_variables)}')
    original_time_sizes = {'tasmin': inputs['tasmin'].sizes['time'], 'tasmax': inputs['tasmax'].sizes['time'], 'pr': inputs['pr'].sizes['time']}
    tasmin_c, tasmax_c, precipitation_in = xr.align(inputs['tasmin'], inputs['tasmax'], inputs['pr'], join='inner', copy=False)
    aligned_time_size = tasmin_c.sizes['time']
    if aligned_time_size == 0:
        raise ValueError('No shared dates were found among tasmin, tasmax, and precipitation.')
    removed_dates = {variable_name: original_size - aligned_time_size for variable_name, original_size in original_time_sizes.items()}
    if any((count > 0 for count in removed_dates.values())):
        print(f'Time alignment retained shared dates only. Removed dates: {removed_dates}')
    tmin_c = select_growing_season(tasmin_c, HULTGREN_SEASON_START, HULTGREN_SEASON_END)
    tmax_c = select_growing_season(tasmax_c, HULTGREN_SEASON_START, HULTGREN_SEASON_END)
    precip_inches = select_growing_season(precipitation_in, HULTGREN_SEASON_START, HULTGREN_SEASON_END)
    if tmin_c.sizes['time'] == 0:
        raise ValueError('No March-August dates remained after alignment.')
    tmin_c, tmax_c, precip_inches = xr.align(tmin_c, tmax_c, precip_inches, join='exact', copy=False)
    daily_gdd_8_31 = bounded_degree_days(tmin_c=tmin_c, tmax_c=tmax_c, lower_c=8.0, upper_c=31.0)
    daily_kdd_31 = degree_days_above_threshold(tmin_c, tmax_c, threshold_c=31.0)
    month = precip_inches.time.dt.month
    corn_phase_1 = precip_inches.where(month == 3, drop=True)
    corn_phase_2 = precip_inches.where(month.isin([4, 5, 6]), drop=True)
    corn_phase_3 = precip_inches.where(month.isin([7, 8]), drop=True)
    soybean_phase_1 = precip_inches.where(month == 3, drop=True)
    soybean_phase_2 = precip_inches.where(month == 4, drop=True)
    soybean_phase_3 = precip_inches.where(month.isin([5, 6]), drop=True)
    soybean_phase_4 = precip_inches.where(month.isin([7, 8]), drop=True)
    variables = {'gdd_8_31C': annual_sum(daily_gdd_8_31), 'kdd_ge31C': annual_sum(daily_kdd_31), 'growing_season_precipitation': annual_sum(precip_inches), 'corn_precip_phase1': annual_sum(corn_phase_1), 'corn_precip_phase2': annual_sum(corn_phase_2), 'corn_precip_phase3': annual_sum(corn_phase_3), 'soybean_precip_phase1': annual_sum(soybean_phase_1), 'soybean_precip_phase2': annual_sum(soybean_phase_2), 'soybean_precip_phase3': annual_sum(soybean_phase_3), 'soybean_precip_phase4': annual_sum(soybean_phase_4)}
    result = xr.Dataset(variables)
    result['gdd_8_31C'].attrs.update(units='degree_Celsius day', long_name='March-August sinusoidal degree days between 8 and 31 degrees Celsius', temporal_aggregation='annual sum from March 1 through August 31', lower_threshold_celsius=8.0, upper_threshold_celsius=31.0)
    result['kdd_ge31C'].attrs.update(units='degree_Celsius day', long_name='March-August sinusoidal degree days above 31 degrees Celsius', temporal_aggregation='annual sum from March 1 through August 31', threshold_celsius=31.0)
    result['growing_season_precipitation'].attrs.update(units='inch', long_name='March-August total precipitation', temporal_aggregation='sum of daily precipitation from March 1 through August 31')
    precipitation_phase_names = ['corn_precip_phase1', 'corn_precip_phase2', 'corn_precip_phase3', 'soybean_precip_phase1', 'soybean_precip_phase2', 'soybean_precip_phase3', 'soybean_precip_phase4']
    phase_descriptions = {'corn_precip_phase1': 'March total precipitation', 'corn_precip_phase2': 'April-June total precipitation', 'corn_precip_phase3': 'July-August total precipitation', 'soybean_precip_phase1': 'March total precipitation', 'soybean_precip_phase2': 'April total precipitation', 'soybean_precip_phase3': 'May-June total precipitation', 'soybean_precip_phase4': 'July-August total precipitation'}
    for variable_name in precipitation_phase_names:
        result[variable_name].attrs.update(units='inch', long_name=phase_descriptions[variable_name], temporal_aggregation='sum of daily precipitation')
    result.attrs.update(paper='Hultgren et al. (2025)', temporal_resolution='annual', growing_season_start=HULTGREN_SEASON_START, growing_season_end=HULTGREN_SEASON_END, gdd_lower_threshold_celsius=8.0, gdd_upper_threshold_celsius=31.0, kdd_threshold_celsius=31.0, temperature_method='sinusoidal daily temperature interpolation', time_alignment='shared dates among tasmin, tasmax, and precipitation', crop_temperature_note='Corn and soybean use the same March-August GDD and KDD indicators.')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    for variable_name in precipitation_phase_names:
        squared_name = f'{variable_name}_squared'
        result[squared_name] = result[variable_name] ** 2
        result[squared_name].attrs.update(units='inch^2', long_name=f'square of county-level {phase_descriptions[variable_name].lower()}', formula=f'{variable_name}^2', calculation_order='phase precipitation aggregated to the county first and squared second')
    result.attrs.update(spatial_resolution='Illinois county', weighting='cosine-latitude grid-cell area correction only; no crop, population, or LandScan weighting', squared_precipitation_calculation='county phase precipitation is calculated first and squared second')
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [ ]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_hultgren,
    overwrite=OVERWRITE,
)

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")